In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "press-t5-qa"
TOWER = 5
TOWER_NAME = "Tower of Press"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 5: Tower of Press


In [2]:
# F1 -- Press Assets

logo_dir = WEB / "images" / "yinyang"
logos = list(logo_dir.glob("yinyang-logo-*.png")) if logo_dir.exists() else []
record("F1-P1", "Multiple logo sizes for press", len(logos) >= 3, f"{len(logos)} logos")

large = [f for f in logos if any(s in f.name for s in ["256", "512", "1024"])]
record("F1-P2", "High-res logos available", len(large) >= 1, f"{len(large)} high-res")

record("F1-P3", "SVG favicon for scalable branding", (WEB / "favicon.svg").exists())

demo = read_file(WEB / "demo.html")
record("F1-P4", "Demo page for press screenshots", len(demo) > 200)

PASS: Multiple logo sizes for press -- 7 logos
PASS: High-res logos available -- 2 high-res
PASS: SVG favicon for scalable branding
PASS: Demo page for press screenshots


In [3]:
# F2 -- Product Description & Messaging

start_html = read_file(WEB / "start.html")
has_desc = bool(re.search(r'<meta[^>]*description[^>]*content', start_html))
record("F2-P1", "Start page has meta description", has_desc)

has_vp = bool(re.search(r"<h1|tagline|hero|welcome", start_html, re.IGNORECASE))
record("F2-P2", "Start page has value proposition", has_vp)

og_pages = 0
for hf in list(WEB.glob("*.html")):
    if re.search(r"og:title", read_file(hf)):
        og_pages += 1
record("F2-P3", "Pages have OG meta for sharing", og_pages >= 5, f"{og_pages} pages")

home_html = read_file(WEB / "home.html")
has_features = bool(re.search(r"feature|automat|recipe|oauth3|evidence", home_html, re.IGNORECASE))
record("F2-P4", "Home page describes key features", has_features)

PASS: Start page has meta description
PASS: Start page has value proposition
PASS: Pages have OG meta for sharing -- 13 pages
PASS: Home page describes key features


In [4]:
# F3 -- Legal & Compliance

record("F3-P1", "LICENSE file exists", (BROWSER_ROOT / "LICENSE").exists())

record("F3-P2", "SECURITY.md exists", (BROWSER_ROOT / "SECURITY.md").exists())

privacy_refs = 0
for hf in list(WEB.glob("*.html")):
    if re.search(r"privacy|cookie|gdpr|data.prot", read_file(hf), re.IGNORECASE):
        privacy_refs += 1
record("F3-P3", "Privacy references across pages", privacy_refs >= 1, f"{privacy_refs} pages")

legal_refs = 0
for hf in list(WEB.glob("*.html")):
    if re.search(r"terms|legal|policy|agreement", read_file(hf), re.IGNORECASE):
        legal_refs += 1
record("F3-P4", "Legal references exist", legal_refs >= 1, f"{legal_refs} pages")

PASS: LICENSE file exists
PASS: SECURITY.md exists
PASS: Privacy references across pages -- 6 pages
PASS: Legal references exist -- 18 pages


In [5]:
# F4 -- Technical Credibility

docs_dir = WEB / "docs"
docs = list(docs_dir.glob("*.html")) if docs_dir.exists() else []
record("F4-P1", "Documentation pages exist", len(docs) >= 1, f"{len(docs)} docs")

tests = list(TESTS.glob("test_*.py")) if TESTS.exists() else []
record("F4-P2", "Test suite exists", len(tests) >= 10, f"{len(tests)} test files")

chain = read_file(SRC / "audit" / "chain.py")
has_chain = bool(re.search(r"sha256|hash|seal", chain, re.IGNORECASE))
record("F4-P3", "Evidence chain exists", has_chain)

oauth3 = list((SRC / "oauth3").glob("*.py")) if (SRC / "oauth3").exists() else []
record("F4-P4", "OAuth3 module (differentiator)", len(oauth3) >= 3, f"{len(oauth3)} files")

PASS: Documentation pages exist -- 3 docs
PASS: Test suite exists -- 107 test files
PASS: Evidence chain exists
PASS: OAuth3 module (differentiator) -- 9 files


In [6]:
# F5 -- Contact & International Reach

# Contact via footer/header links across HTML pages
contact_pages = 0
for hf in list(WEB.glob("*.html")):
    content = read_file(hf)
    if re.search(r"support|contact|feedback|help", content, re.IGNORECASE):
        contact_pages += 1
record("F5-P1", "Support/contact references across pages", contact_pages >= 3,
       f"{contact_pages} pages with support/contact refs")

glossary = read_file(WEB / "glossary.html")
record("F5-P2", "Glossary for press terminology", len(glossary) > 200)

guide = read_file(WEB / "guide.html")
record("F5-P3", "Getting started guide", len(guide) > 200)

locale_dir = BROWSER_ROOT / "app" / "locales" / "yinyang"
locales = len(list(locale_dir.glob("*.json"))) if locale_dir.exists() else 0
record("F6-P1", "Multi-language support", locales >= 10, f"{locales} locales")

# i18n in layout.js or solace.js
layout_js = read_file(WEB / "js" / "layout.js")
solace_js = read_file(WEB / "js" / "solace.js")
has_i18n = bool(re.search(r"i18n|locale|language|data-i18n", layout_js + solace_js, re.IGNORECASE))
record("F6-P2", "i18n support in JS", has_i18n)

themes = list((WEB / "css" / "themes").glob("*.css")) if (WEB / "css" / "themes").exists() else []
record("F6-P3", "Multiple themes (accessibility story)", len(themes) >= 2, f"{len(themes)} themes")

PASS: Support/contact references across pages -- 12 pages with support/contact refs
PASS: Glossary for press terminology
PASS: Getting started guide
PASS: Multi-language support -- 47 locales
PASS: i18n support in JS
PASS: Multiple themes (accessibility story) -- 3 themes


In [7]:
# EVIDENCE SUMMARY
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
conv = "YES" if summary["converged"] else "NO"
ehash = summary['evidence_hash']
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {conv}")
print(f"Evidence hash: {ehash}")
if summary["findings"]:
    print()
    print("FINDINGS:")
    for fi in summary["findings"]:
        fid = fi['id']
        fname = fi['name']
        fdetail = fi.get('detail', '')
        print(f"  [{fid}] {fname}: {fdetail}")
print()
print(json.dumps(summary, indent=2))


TOWER 5: Tower of Press
Probes: 22 | Passed: 22 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 445c5e9801ada110

{
  "surface": "press-t5-qa",
  "tower": 5,
  "tower_name": "Tower of Press",
  "timestamp": "2026-03-06T11:28:32.341303",
  "total_probes": 22,
  "passed": 22,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "445c5e9801ada110"
}
